In [26]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already exists — pulling latest changes
Already up to date.


In [27]:
# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python Challenge/compile_cython.py
os.chdir(WORKING_DIR)

running build_ext
copying build/lib.linux-x86_64-cpython-312/CFW_D_Similarity_Cython_SGD.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/CFW_DVV_Similarity_Cython_SGD.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/FBSM_Rating_Cython_SGD.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/HP3_Similarity_Cython_SGD.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/Compute_Similarity_Cython.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/Triangular_Matrix.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/SLIM_BPR_Cython_Epoch.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/Sparse_Matrix_Tree_CSR.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312/MatrixFactorization_Cython_Epoch.cpython-312-x86_64-linux-gnu.so -> 
copying build/lib.linux-x86_64-cpython-312

In [28]:
%%capture
if not IS_LOCAL:
    !pip install optuna

import optuna

In [29]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on colab — storage at: /content/drive/MyDrive/RecSys


In [30]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [31]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0

    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]

        if len(relevant_items)>0:
            num_eval+=1

            recommended_items = recommender.recommend(user_id, cutoff=at)

            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# Train a KNN for each similarity

["cosine", "pearson", "jaccard", "tanimoto", "adjusted", "euclidean"]

In [34]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

In [38]:
def perform_optimization(similarity, n_trials):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity

    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", 10, 1500),
            shrink=optuna_trial.suggest_int("shrink", 0, 2000),
            normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
            feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
        )

        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

## Pearson

In [39]:
SIMILARITY = "pearson"

In [40]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

  0%|          | 0/100 [00:00<?, ?it/s]

Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 657.66 column/sec. Elapsed time 10.60 sec
[I 2025-11-08 15:18:32,451] Trial 5 finished with value: 0.0032129050232470036 and parameters: {'topK': 239, 'shrink': 700, 'normalize': True, 'feature_weighting': 'none'}. Best is trial 5 with value: 0.0032129050232470036.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 516.67 column/sec. Elapsed time 13.49 sec
[I 2025-11-08 15:19:04,832] Trial 6 finished with value: 0.1450703889131546 and parameters: {'topK': 361, 'shrink': 336, 'normalize': True, 'feature_weighting': 'BM25'}. Best is trial 6 with value: 0.1450703889131546.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 672.56 column/sec. Elapsed time 10.36 sec
[I 2025-11-08 15:19:35,211] Trial 7 finished with value: 0.0032129050232470036 and parameters: {'topK': 791, 'shrink': 1013, 'normalize': False, 'f

In [41]:
optuna.visualization.plot_optimization_history(optuna_study)

In [42]:
optuna.visualization.plot_param_importances(optuna_study)

In [43]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [45]:
STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

def pearson_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = ItemKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 0, 100),
        shrink=optuna_trial.suggest_int("shrink", 0, 100),
        normalize=True,
        feature_weighting="TF-IDF"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    pearson_tuning_function,
    study_name=STUDY_NAME,
    n_trials=20
)

  0%|          | 0/20 [00:00<?, ?it/s]

Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 661.82 column/sec. Elapsed time 10.53 sec
[I 2025-11-08 16:45:25,813] Trial 20 finished with value: 0.20579294860363007 and parameters: {'topK': 21, 'shrink': 97}. Best is trial 13 with value: 0.20775166153907776.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 667.71 column/sec. Elapsed time 10.44 sec
[I 2025-11-08 16:45:51,593] Trial 21 finished with value: 0.20722205936908722 and parameters: {'topK': 56, 'shrink': 62}. Best is trial 13 with value: 0.20775166153907776.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 717.12 column/sec. Elapsed time 9.72 sec
[I 2025-11-08 16:46:15,835] Trial 22 finished with value: 0.2072538584470749 and parameters: {'topK': 41, 'shrink': 48}. Best is trial 13 with value: 0.20775166153907776.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity co

### **Best Model**
- Best Value: 0.21077559888362885
- Best Params: {'topK': 11, 'shrink': 3, 'normalize': True, 'feature_weighting': 'TF-IDF'}